In [5]:
from langchain.tools import tool
from langchain_community.chat_models import ChatZhipuAI  # GLM专用类
from langchain.chat_models import init_chat_model


model = ChatZhipuAI(
    model_name="glm-4.7", 
    temperature=0,
    max_tokens=2048,
    api_key="51d9189503ad476fbba1c56e14e60826.Cckh0az1l4lljZNe"
)


# Define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


# Augment the LLM with tools
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)


from langgraph.graph import add_messages
from langchain.messages import (
    SystemMessage,
    HumanMessage,
    ToolCall,
)
from langchain_core.messages import BaseMessage
from langgraph.func import entrypoint, task

1. 定义模型节点
模型节点用于调用LLM，并决定是否调用某个工具。

In [6]:
@task
def call_llm(messages: list[BaseMessage]):
    """LLM决定是否调用工具"""
    return model_with_tools.invoke(
        [
            SystemMessage(
                content="你是一个乐于助人的助手，负责对一组输入执行算术。"
            )
        ]
        + messages
    )

2. 定义工具节点
工具节点用于调用工具并返回结果。

In [7]:
@task
def call_tool(tool_call: ToolCall):
    """Performs the tool call"""
    tool = tools_by_name[tool_call["name"]]
    return tool.invoke(tool_call)

3. 定义代理人
该代理是通过@entrypoint功能

在函数式API中，你不是明确定义节点和边，而是在单一函数内编写标准的控制流逻辑（循环、条件句）。

In [8]:
@entrypoint()
def agent(messages: list[BaseMessage]):
    model_response = call_llm(messages).result()

    while True:
        if not model_response.tool_calls:
            break

        # Execute tools
        tool_result_futures = [
            call_tool(tool_call) for tool_call in model_response.tool_calls
        ]
        tool_results = [fut.result() for fut in tool_result_futures]
        messages = add_messages(messages, [model_response, *tool_results])
        model_response = call_llm(messages).result()

    messages = add_messages(messages, model_response)
    return messages


# Invoke
messages = [HumanMessage(content="计算12+8")]
for chunk in agent.stream(messages, stream_mode="updates"):
    print(chunk)
    print("\n")

{'call_llm': AIMessage(content='我来帮您计算12+8。', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":12,"b":8}', 'name': 'add'}, 'id': 'call_-7806872203223363611', 'index': 0, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_tokens': 368, 'prompt_tokens_details': {'cached_tokens': 366}, 'total_tokens': 431}, 'model_name': 'glm-4.7', 'finish_reason': 'tool_calls'}, id='lc_run--019cf0a4-2ac6-7c91-96bb-c0bb2c99d402-0', tool_calls=[{'name': 'add', 'args': {'a': 12, 'b': 8}, 'id': 'call_-7806872203223363611', 'type': 'tool_call'}], invalid_tool_calls=[])}


{'call_tool': ToolMessage(content='20', name='add', id='6f6b7b9e-5d0d-4295-bb03-42c1adc0e1ec', tool_call_id='call_-7806872203223363611')}


{'call_llm': AIMessage(content='12 + 8 = 20', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_t